### Installing Dependencies

In [3]:
!pip install Unidecode ldap3 pandas numpy faker pyarrow

  Using cached Unidecode-1.4.0-py3-none-any.whl.metadata (13 kB)
  Using cached ldap3-2.9.1-py2.py3-none-any.whl.metadata (5.4 kB)
  Using cached faker-37.12.0-py3-none-any.whl.metadata (15 kB)
  Using cached pyarrow-22.0.0-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
Using cached Unidecode-1.4.0-py3-none-any.whl (235 kB)
Using cached ldap3-2.9.1-py2.py3-none-any.whl (432 kB)
Using cached faker-37.12.0-py3-none-any.whl (2.0 MB)
Using cached pyarrow-22.0.0-cp313-cp313-win_amd64.whl (28.0 MB)
Using cached pyasn1-0.6.1-py3-none-any.whl (83 kB)

   ---------------------------------------- 0/5 [Unidecode]
   ---------------------------------------- 0/5 [Unidecode]
   ---------------------------------------- 0/5 [Unidecode]
   ---------------------------------------- 0/5 [Unidecode]
   ---------------------------------------- 0/5 [Unidecode]
   ---------------------------------------- 0/5 [Unidecode]
   ---------------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
!pip install --upgrade pyarrow pandas



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import unidecode
import re
import os
from faker import Faker
from datetime import datetime
import pyarrow.parquet as pq


#### Funções Utilitárias

In [5]:
# Inicializa o Faker para gerar dados fictícios
fake = Faker('pt_BR')

# --- Funções de I/O (Simulando S3 localmente) ---
# Vamos criar pastas 'data/bronze' e 'data/silver' para simular os buckets

def setup_directories():
    """Cria as pastas locais para simular o S3."""
    os.makedirs('data/acme_corp/bronze/active_directory', exist_ok=True)
    os.makedirs('data/acme_corp/silver/active_directory_it_users', exist_ok=True)
    os.makedirs('data/acme_corp/auxiliary_data/locations', exist_ok=True)
    os.makedirs('data/acme_corp/auxiliary_data/master_data', exist_ok=True)

def write_parquet_local(df, path):
    """Função simplificada para 'upload' (salvar localmente)."""
    print(f"Salvando dados em: {path}")
    df.to_parquet(
        path,
        index=False,
        engine="pyarrow",
        coerce_timestamps="ms",
        use_deprecated_int96_timestamps=False
    )

def read_parquet_local(path):
    print(f"Lendo dados de: {path}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo não encontrado: {path}. Rode a simulação de dados primeiro.")

    table = pq.read_table(path)
    table = table.replace_schema_metadata({})
    return table.to_pandas(ignore_metadata=True)

# Roda a configuração
setup_directories()

#### Geração de dados

In [6]:
def generate_fake_bronze_data(num_users=200):
    """
    Gera um DataFrame do Active Directory com dados "sujos" e problemas 
    para serem corrigidos na camada Silver.
    """
    data = []
    
    # Lista de países "sujos" que o código Silver corrige
    paises_sujos = [
        'Brasil', 'brasil', 'Estados Unidos', 'United States', 
        'Alemanha', 'Africa do Sul', 'África do Sul', 'Russia', 
        'Turquia', 'Espana', 'Mexico', None
    ]

    # Cidades correspondentes (algumas com acento)
    cidades = [
        'São Paulo', 'Jaraguá do Sul', 'New York', 'Washington', 
        'Berlin', 'Cape Town', 'Cape Town', 'Moscow', 
        'Istanbul', 'Madrid', 'Mexico City', 'Blumenau'
    ]
    
    # Status de conta (ativos, desativados)
    # 512, 544, 66048 = Ativos (queremos manter)
    # 514, 546 = Desativados (queremos filtrar)
    user_account_controls = ['512', '544', '66048', '514', '546', '66050']
    
    # Tipo de licença (colaborador)
    # 001, 002, 003, 004 = Tipos válidos
    # 999, 000 = Tipos inválidos (queremos filtrar)
    car_licenses = ['001', '002', '003', '004', '999', '000', '001']

    for i in range(num_users):
        name = fake.name()
        first_name = name.split(' ')[0]
        last_name = name.split(' ')[-1]
        
        # Formatos de gestor "sujos"
        manager_dn = [
            f"CN={fake.name()},OU=Users,OU=Brazil,DC=acme,DC=com",
            f"CN={fake.name().replace(' ', r' \ ')},OU=Users,DC=acme,DC=com", # Com escape
            "CN=Desativada-IT,OU=Disabled,DC=acme,DC=com",
            "CN=,OU=Users,DC=acme,DC=com", # Vazio
            "{}",
            None
        ]
        
        idx = i % len(paises_sujos)
        
        data.append({
            'displayName': name,
            'sn': last_name,
            'givenName': first_name,
            'sAMAccountName': f"{first_name[0].lower()}{last_name.lower()}{fake.numerify('##')}",
            'mail': f"{first_name.lower()}.{last_name.lower()}@acme.com",
            'employeeID': fake.unique.numerify('#######'),
            'company': 'ACME Corp',
            'department': fake.job().split(' ')[-1],
            'title': fake.job(),
            
            # --- Colunas-problema ---
            'co': paises_sujos[idx], # País (com acento, maiuscula, etc)
            'l': cidades[idx] if np.random.rand() > 0.1 else None, # Cidade (pode ser nula)
            'userAccountControl': np.random.choice(user_account_controls), # Status
            'carLicense': np.random.choice(car_licenses), # Tipo de colaborador
            'manager': np.random.choice(manager_dn) # Gestor em formato DN
        })
        
    df = pd.DataFrame(data)
    
    # Adiciona colunas que seu script original buscava mas não usava
    # Apenas para simular a "extração" completa
    df['description'] = 'Usuário de TI'
    df['division'] = 'IT'
    df['physicalDeliveryOfficeName'] = 'Sala ' + fake.numerify('###')
    
    snapshot_date = datetime.now().strftime("%Y-%m-%d")
    df["execution_date"] = snapshot_date
    
    return df

In [ ]:
# UN/LOCODE + ISO 3166-1
def generate_fake_auxiliary_data():
    """Gera os datasets auxiliares (location e master) com dados limpos."""
    
    # --- 1. Location Data (world_country_city_location) ---
    # Deve conter os dados "limpos" que esperamos após a transformação
    location_data = [
        # Dados que vão bater com 'co' (país)
        {'country_code': 'BR', 'country_name': 'brazil', 'city_name': 'sao paulo', 'continent': 'South America', 'latitude': -23.55, 'longitude': -46.63, 'site': 'ACME Brazil SP'},
        {'country_code': 'BR', 'country_name': 'brazil', 'city_name': 'jaragua do sul', 'continent': 'South America', 'latitude': -26.48, 'longitude': -49.06, 'site': 'ACME Brazil HQ'},
        {'country_code': 'US', 'country_name': 'united states of america', 'city_name': 'new york', 'continent': 'North America', 'latitude': 40.71, 'longitude': -74.00, 'site': 'ACME USA East'},
        {'country_code': 'DE', 'country_name': 'germany', 'city_name': 'berlin', 'continent': 'Europe', 'latitude': 52.52, 'longitude': 13.40, 'site': 'ACME Germany'},
        {'country_code': 'ZA', 'country_name': 'south africa', 'city_name': 'cape town', 'continent': 'Africa', 'latitude': -33.92, 'longitude': 18.42, 'site': 'ACME South Africa'},
        {'country_code': 'RU', 'country_name': 'russian federation', 'city_name': 'moscow', 'continent': 'Europe', 'latitude': 55.75, 'longitude': 37.61, 'site': 'ACME Russia'},
        {'country_code': 'TR', 'country_name': 'turkey', 'city_name': 'istanbul', 'continent': 'Asia', 'latitude': 41.00, 'longitude': 28.97, 'site': 'ACME Turkey'},
        {'country_code': 'ES', 'country_name': 'spain', 'city_name': 'madrid', 'continent': 'Europe', 'latitude': 40.41, 'longitude': -3.70, 'site': 'ACME Spain'},
        
        # Dados que vão bater com 'l' (cidade) quando 'co' for nulo
        {'country_code': 'BR', 'country_name': 'brazil', 'city_name': 'blumenau', 'continent': 'South America', 'latitude': -26.91, 'longitude': -49.06, 'site': 'ACME Brazil SC'},
        
        # Duplicata para testar o drop_duplicates
        {'country_code': 'BR', 'country_name': 'brazil', 'city_name': 'sao paulo', 'continent': 'South America', 'latitude': -23.55, 'longitude': -46.63, 'site': 'ACME Brazil SP (Duplicado)'}
    ]
    df_location = pd.DataFrame(location_data)
    
    # Path
    LOCATION_KEY = 'data/acme_corp/auxiliary_data/locations/world_country_city_location.parquet'
    write_parquet_local(df_location, LOCATION_KEY)
    print(f"Dados de localização gerados em {LOCATION_KEY}")

    
    # --- 2. Master Data ---
    master_data = [
        {'country': 'brazil', 'region': 'LATAM', 'region_manager': 'Carlos Silva'},
        {'country': 'united states of america', 'region': 'NA', 'region_manager': 'John Smith'},
        {'country': 'germany', 'region': 'EMEA', 'region_manager': 'Hans Mueller'},
        {'country': 'south africa', 'region': 'EMEA', 'region_manager': 'Hans Mueller'},
        {'country': 'russia', 'region': 'APAC', 'region_manager': 'Olga Petrova'}, # Note o 'russia'
        {'country': 'turkey', 'region': 'EMEA', 'region_manager': 'Hans Mueller'},
        {'country': 'spain', 'region': 'EMEA', 'region_manager': 'Hans Mueller'}
    ]
    df_master = pd.DataFrame(master_data)
    
    MASTER_DATA_KEY = 'data/acme_corp/auxiliary_data/master_data/master_data.parquet'
    write_parquet_local(df_master, MASTER_DATA_KEY)
    print(f"Dados Mestres gerados em {MASTER_DATA_KEY}")

# Executa a geração dos dados auxiliares
generate_fake_auxiliary_data()

#### Bronze Layer Construction

In [75]:
# --- SIMULAÇÃO DA CAMADA BRONZE ---
# Em um pipeline real, isso seria o fim da sua função 'bronze_layer_construction'

print("Executando a extração Bronze (Simulada)...")
snapshot_date = datetime.now().strftime("%Y-%m-%d")

# 1. Gerar os dados falsos
bronze_ad_data = generate_fake_bronze_data(num_users=500)

# 2. Caminho do S3
BRONZE_AD_PATH = f'data/acme_corp/bronze/active_directory/{snapshot_date}.parquet'

# 3. Salvar o arquivo parquet localmente (simulando o upload)
write_parquet_local(bronze_ad_data, BRONZE_AD_PATH)

print("\n--- Dados Brutos (Bronze) Gerados (Amostra): ---")
bronze_ad_data[['displayName', 'co', 'l', 'userAccountControl', 'carLicense', 'manager']].head()

Executando a extração Bronze (Simulada)...
Salvando dados em: data/acme_corp/bronze/active_directory/2025-11-09.parquet

--- Dados Brutos (Bronze) Gerados (Amostra): ---


,displayName,co,l,userAccountControl,carLicense,manager
0,Diogo Souza,Brasil,São Paulo,546,002,"CN=,OU=Users,DC=acme,DC=com"
1,Amanda Leão,brasil,Jaraguá do Sul,546,001,"CN=Desativada-IT,OU=Disabled,DC=acme,DC=com"
2,Vitória Fogaça,Estados Unidos,New York,546,999,"CN=Sr. \ João \ Gabriel \ Rocha,OU=Users,DC=ac..."
3,Caio Gomes,United States,Washington,546,003,"CN=Desativada-IT,OU=Disabled,DC=acme,DC=com"
4,Carlos Eduardo Cunha,Alemanha,Berlin,66050,004,"CN=Maria \ Laura \ da \ Costa,OU=Users,DC=acme..."


### Transformações da camada SILVER

In [76]:
# --- INÍCIO DO SCRIPT SILVER ---

print("Iniciando a construção da Camada Silver...")
snapshot_date = datetime.now().strftime("%Y-%m-%d")

# 1. Carregar dados do Active Directory (do nosso arquivo local)
BRONZE_AD_PATH = f'data/acme_corp/bronze/active_directory/{snapshot_date}.parquet'

active_dir = read_parquet_local(BRONZE_AD_PATH)
print(f"Dados brutos do AD carregados. Total de registros: {len(active_dir)}")


Iniciando a construção da Camada Silver...
Lendo dados de: data/acme_corp/bronze/active_directory/2025-11-09.parquet
Dados brutos do AD carregados. Total de registros: 500


#### Bloco 1 - Filtragem e Mapeamento de Status

In [ ]:
# Garantir que as colunas de filtro sejam string
active_dir['userAccountControl'] = active_dir['userAccountControl'].astype(str)
active_dir['carLicense'] = active_dir['carLicense'].astype(str)

active_user_codes = ['512', '544', '66048'] # Filtros para contas ativas
valid_collaborator_codes = ['001', '002', '003', '004'] # Filtros para tipos de colaboradores

# Aplicando os filtros
active_dir_filtered = active_dir[
    active_dir['userAccountControl'].isin(active_user_codes) &
    active_dir['carLicense'].isin(valid_collaborator_codes)
].copy() # Cria um novo DataFrame independente, sem ligação de memória com o original

print(f"Registros após filtro: {len(active_dir_filtered)} (Removidos: {len(active_dir) - len(active_dir_filtered)})")

# --- Mapeamento de Códigos ---
# Agora, traduzimos os códigos para valores legíveis
type_map = {
    '001': 'Employee',
    '002': 'Intern',
    '003': 'Third Parties',
    '004': 'Generic'
}
active_dir_filtered['type_of_collaborator'] = active_dir_filtered['carLicense'].map(type_map)

status_map = {
    '512': 'Active User',
    '544': 'Active User and password not required',
    '66048': 'Active User and password not expire',
}
active_dir_filtered['user_status'] = active_dir_filtered['userAccountControl'].map(status_map)

print("\nAmostra após Mapeamento de Status:")
active_dir_filtered[['displayName', 'type_of_collaborator', 'user_status']].head()


Registros após filtro: 193 (Removidos: 307)

Amostra após Mapeamento de Status:


,displayName,type_of_collaborator,user_status
5,Bárbara Gomes,Generic,Active User and password not expire
6,Luiz Fernando Oliveira,Employee,Active User and password not required
10,Dr. Apollo Montenegro,Intern,Active User
11,Emilly Barros,Third Parties,Active User and password not expire
12,Sabrina Melo,Generic,Active User and password not expire


#### Bloco 2 - Limpeza e Padronização (Dados do AD)

In [78]:
# --- Bloco 2: Limpeza e Padronização de Localização (AD) ---

# 1. Mapear valores inconsistentes de países
map_countries = {
    'Brasil': 'Brazil',
    'Estados Unidos': 'United States of America',
    'United States': 'United States of America',
    'Alemanha': 'Germany',
    'Africa do Sul': 'South Africa',
    'África do Sul': 'South Africa',
    'Egito': 'Egypt', # Não estava nos dados fake, mas mantemos
    'Russia': 'Russian Federation',
    'Equador': 'Ecuador', # Não estava nos dados fake, mas mantemos
    'Turquia': 'Turkey',
    'Espana': 'Spain'
}

# Usamos .loc[:, 'co'] para modificar o DataFrame original de forma segura
active_dir_filtered.loc[:, 'co'] = active_dir_filtered['co'].replace(map_countries)

# 2. Padronização: Remover acentos, converter para minúsculas e remover espaços
def standardize_text(x):
    if isinstance(x, str):
        return unidecode.unidecode(x).lower().strip()
    return x

active_dir_filtered.loc[:, 'co'] = active_dir_filtered['co'].apply(standardize_text)
active_dir_filtered.loc[:, 'l'] = active_dir_filtered['l'].apply(standardize_text)

print("\nAmostra de 'co' (país) após padronização:")
print(active_dir_filtered[active_dir_filtered['co'].notna()]['co'].value_counts().head())


Amostra de 'co' (país) após padronização:
co
united states of america    38
south africa                35
spain                       18
germany                     16
mexico                      15
Name: count, dtype: int64


#### Bloco 3 - Carregando e Padronizando Dados Auxiliares

In [79]:
LOCATION_KEY = 'data/acme_corp/auxiliary_data/locations/world_country_city_location.parquet'
full_location = read_parquet_local(LOCATION_KEY)

print(f"Dados de localização carregados. Total: {len(full_location)} registros.")

# 1. Aplicar a MESMA padronização nos dados de localização
full_location.loc[:, 'country_name'] = full_location['country_name'].apply(standardize_text)
full_location.loc[:, 'city_name'] = full_location['city_name'].apply(standardize_text)

# 2. Remover duplicatas (ex: se 'sao paulo' aparece 2x)
# Mantemos 'first' para garantir consistência
full_location = full_location.drop_duplicates(subset='city_name', keep='first')
print(f"Dados de localização após remover duplicatas: {len(full_location)} registros.")

# 3. Criar uma visão "apenas país" para o primeiro join
full_location_country = full_location[
    ['country_code', 'country_name', 'continent', 'latitude', 'longitude', 'site']
].copy()
full_location_country = full_location_country.drop_duplicates(subset=['country_code', 'country_name'])

print(f"\nVisão de Países (para Join 1): {len(full_location_country)} registros.")
print(f"Visão de Cidades (para Join 2): {len(full_location)} registros.")

Lendo dados de: data/acme_corp/auxiliary_data/locations/world_country_city_location.parquet
Dados de localização carregados. Total: 10 registros.
Dados de localização após remover duplicatas: 9 registros.

Visão de Países (para Join 1): 7 registros.
Visão de Cidades (para Join 2): 9 registros.


#### Bloco 4 - Lógica de Join Complexa (Enriquecimento)

In [80]:
# --- Bloco 4: Lógica de Join Complexa (Enriquecimento de Localização) ---

# 4a. Tentativa 1: Join Principal por País (co)
print("Executando Join 1 (por País)...")
df_merged = active_dir_filtered.merge(
    full_location_country,
    how='left',
    left_on='co',           # Coluna País do AD
    right_on='country_name', # Coluna País da ONU/ISO
    suffixes=('', '_country') 
)

# 4b. Isolando Falhas: Registros que falharam no Join por País (tiveram NaN)
null_country_mask = df_merged['co'].isna() | df_merged['country_name'].isna()
df_failed_join_1 = df_merged[null_country_mask].copy()

# 4c. Tentativa 2: Join de Resgate por Cidade (l)
print(f"Join 1 falhou para {len(df_failed_join_1)} registros. Tentando Join 2 (por Cidade)...")

# Remove colunas nulas do primeiro join para evitar conflito no próximo merge
cols_to_drop = [col for col in df_failed_join_1.columns if col.endswith('_country') or col in full_location_country.columns]
df_failed_join_1 = df_failed_join_1.drop(columns=cols_to_drop, errors='ignore')

# Tenta preencher a localização usando a Cidade (l) como chave. Usa a tabela 'full_location'.
df_city_join = df_failed_join_1.merge(
    full_location, # Tabela de localização COMPLETA (país + cidade)
    how='left',
    left_on='l', # Coluna Cidade do AD
    right_on='city_name'
)

# 4d. Preenchimento (Back-filling) -> REMOVIDO
# (Comentário removido, pois a linha foi removida do script)

# 4e. Re-combinação dos DataFrames
# 1. Isola os registros que tiveram sucesso na Tentativa 1
df_success_join_1 = df_merged[~null_country_mask]

# 2. Seleciona as colunas do df_merged nos registros resgatados para permitir o concat
df_rescued_join_2 = df_city_join[df_merged.columns] 

# 3. Concatena (empilha) os sucessos do Join 1 e os resgatados do Join 2
active_dir_with_location = pd.concat(
    [df_success_join_1, df_rescued_join_2],
    ignore_index=True
)

print(f"\nTotal de registros após enriquecimento: {len(active_dir_with_location)}")
print("Amostra de dados enriquecidos (veja 'l', 'co' e 'continent'):")
active_dir_with_location[['displayName', 'l', 'co', 'continent', 'site']].sample(5)

Executando Join 1 (por País)...
Join 1 falhou para 44 registros. Tentando Join 2 (por Cidade)...

Total de registros após enriquecimento: 193
Amostra de dados enriquecidos (veja 'l', 'co' e 'continent'):


,displayName,l,co,continent,site
191,Anna Liz Moreira,blumenau,None,South America,ACME Brazil SC
93,Agatha Nunes,None,spain,Europe,ACME Spain
71,Dra. Joana Ribeiro,berlin,germany,Europe,ACME Germany
23,Ravi Lucca Castro,new york,united states of america,North America,ACME USA East
158,Ísis Barbosa,mexico city,mexico,NaN,NaN


#### Bloco 5 - Join Final e Limpezas

In [81]:
# 1. Carregar Master Data
MASTER_DATA_KEY = 'data/acme_corp/auxiliary_data/master_data/master_data.parquet'
master_data = read_parquet_local(MASTER_DATA_KEY)

print(f"Dados Mestres carregados: {len(master_data)} registros.")

# 2. Join com Master Data
act_dir_transformed = active_dir_with_location.merge(
    master_data,
    how='left',
    left_on='country_name',
    right_on='country',
    suffixes=('', '_map')
)

# Limpa coluna duplicada do join
act_dir_transformed = act_dir_transformed.drop(columns=['country'], errors='ignore')

# 3. Limpezas Finais
# Padronizar 'russian federation' -> 'russia'
map_countries_final = {
    'russian federation': 'russia',
}
act_dir_transformed.loc[:, 'country_name'] = act_dir_transformed['country_name'].replace(map_countries_final)

# Formato 'Title' (Primeira Maiúscula) para apresentação
act_dir_transformed['country_name'] = act_dir_transformed['country_name'].str.title()

print("\nAmostra após Join com Master Data (veja 'region'):")
act_dir_transformed[['displayName', 'country_name', 'continent', 'region']].sample(5)

Lendo dados de: data/acme_corp/auxiliary_data/master_data/master_data.parquet
Dados Mestres carregados: 7 registros.

Amostra após Join com Master Data (veja 'region'):


,displayName,country_name,continent,region
143,Rafael da Cruz,Brazil,South America,LATAM
151,Luna Oliveira,Brazil,South America,LATAM
52,Dr. Bernardo Vasconcelos,Brazil,South America,LATAM
179,Ana Júlia Novaes,Brazil,South America,LATAM
24,Rafael Costela,United States Of America,North America,NA


#### Bloco 6 - Extração de Gestor com Regex

In [82]:
def parse_manager(x):
    """
    Usa Regex para extrair o nome do gestor de um DN (Distinguished Name) do LDAP.
    Retorna (nome_do_gestor, status_do_gestor).
    """
    # 1. Tratar nulos ou strings vazias
    if pd.isna(x) or str(x).strip() in ["", "{}"]:
        return None, None
    
    x = str(x).strip()
    
    # 2. Tratar casos específicos (contas inválidas ou desativadas)
    if x.startswith("CN=,"): # Gestor mal preenchido
        return None, None
    if x.startswith("CN=Desativada"):
        return None, "Disabled"
    
    # 3. A Mágica do Regex:
    #    r"CN=([^,]+)"
    #    CN=      -> Procura o texto literal "CN="
    #    (        -> Inicia um "grupo de captura" (o que queremos extrair)
    #    [^,]     -> Procura qualquer caractere EXCETO (^) uma vírgula (,)
    #    +        -> ... uma ou mais vezes
    #    )        -> Fecha o grupo de captura
    match = re.match(r"CN=([^,]+)", x)
    
    if match:
        # match.group(1) retorna o texto capturado pelo primeiro grupo (parênteses)
        manager_name = match.group(1)
        # Se o nome tiver barras de escape (ex: 'Fulano \ de \ Tal'), remove
        manager_name = manager_name.replace(r' \ ', ' ')
        return manager_name, "Active"
    
    # 4. Se nada der match, retorna nulo
    return None, None

# Aplicando a função
# Isso cria uma lista de tuplas: [('Nome1', 'Status1'), ('Nome2', 'Status2'), ...]
parsed_results = [parse_manager(x) for x in act_dir_transformed["manager"]]

# 'zip(*...)' é um truque para "desagrupar" as tuplas em duas listas separadas
manager_names, manager_status = zip(*parsed_results)

# Atribuir as novas colunas
act_dir_transformed["manager_extracted"] = manager_names
act_dir_transformed["manager_status"] = manager_status

print("\nAmostra de Extração de Gestores (Regex):")
act_dir_transformed[['manager', 'manager_extracted', 'manager_status']].sample(5)


Amostra de Extração de Gestores (Regex):


,manager,manager_extracted,manager_status
157,"CN=Zoe \ da \ Mata,OU=Users,DC=acme,DC=com",Zoe da Mata,Active
83,"CN=Lucca Rodrigues,OU=Users,OU=Brazil,DC=acme,...",Lucca Rodrigues,Active
154,{},None,None
159,"CN=Srta. Milena Gomes,OU=Users,OU=Brazil,DC=ac...",Srta. Milena Gomes,Active
7,"CN=Elisa Alves,OU=Users,OU=Brazil,DC=acme,DC=com",Elisa Alves,Active


#### Bloco 7 - Lógica Final e Salvamento

In [83]:
# Simulação de lógicas de negócio adicionais
def cost_center_fixing(df):
    """
    Função 'stub' (simulada) para representar outras lógicas de negócio
    que poderiam ser aplicadas, como correção de centro de custo.
    """
    print("Aplicando lógica de correção de Centro de Custo (Simulado)...")
    # Ex: df.loc[df['department'] == 'TI', 'cost_center'] = '9001'
    # Para este exemplo, apenas retornamos o df
    return df

df_final_silver = cost_center_fixing(act_dir_transformed)

# Adicionar data de execução
df_final_silver["execution_date"] = snapshot_date

# --- Salvando na Camada Silver ---
SILVER_AD_PATH = f'data/acme_corp/silver/active_directory_it_users/{snapshot_date}.parquet'

write_parquet_local(df_final_silver, SILVER_AD_PATH)

print("\n--- 🚀 Processo Concluído! ---")
print(f"Dados Silver salvos em: {SILVER_AD_PATH}")
print(f"Total de registros Silver: {len(df_final_silver)}")

print("\nAmostra Final do DataFrame Silver:")
df_final_silver.info()

Aplicando lógica de correção de Centro de Custo (Simulado)...
Salvando dados em: data/acme_corp/silver/active_directory_it_users/2025-11-09.parquet

--- 🚀 Processo Concluído! ---
Dados Silver salvos em: data/acme_corp/silver/active_directory_it_users/2025-11-09.parquet
Total de registros Silver: 193

Amostra Final do DataFrame Silver:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   displayName                 193 non-null    object 
 1   sn                          193 non-null    object 
 2   givenName                   193 non-null    object 
 3   sAMAccountName              193 non-null    object 
 4   mail                        193 non-null    object 
 5   employeeID                  193 non-null    object 
 6   company                     193 non-null    object 
 7   department                  193 non-null